In [1]:
import math

## Binomial Option Pricing

This notebook implements the one-period and multi-period binomial option pricing as a discrete-time foundation for later Black–Scholes and delta-hedging analysis.Binomial model prices an option by replicating its payoff in discrete time. A call option allows owner to buy security on/before maturity date; a put option allows owner to sell on/before maturity date

We define:

- $S_0$: initial stock price
- u: up multiplier
- d: down multiplier
- K: strike price
- r: gross risk-free rate
- $C_u$: call payoff in up state
- $C_d$: call payoff in down state

Let:

In [ ]:
S0 = 100
u = 1.2
d = 0.8
K = 100
r = 1.05
n = 3

At maturity T, payoff:

- $C_T$ = Max[0, $S_T$ - K]
- $P_T$ = Max[0, K - $S_T$]


Consider a Bernoulli trial; call option can either increase to $C_u$ or decrease to $C_d$.

After one period, payoff:
- $C_u$ = max(u$S_0$ - K, 0)
- $C_d$ = max(d$S_0$ - K, 0)

In [8]:
Cu = max(u * S0 - K, 0)
Cd = max(d * S0 - K, 0)

Consider a portfolio of stocks & bonds today, with $\Delta$ shares of stocks, and B dollars of bonds
Total cost today: $V_0$ = $S_0$ $\Delta$ + B. For portfolio payoff tomorrow $V_1$ has options:
- u $S_0$ $\Delta$ + rB
- d $S_0$ $\Delta$ + rB

Choose $\Delta$ and B s.t. u $S_0$ + rB = $C_u$ and d $S_0$ + rB = $C_d$

Using simultaneous equations, we can solve for $\Delta^*$ and $B^*$
- $\Delta^* = \frac{C_u - C_d}{S_0(u-d)}$
- $B^* = \frac{uC_d - dC_u}{r(u-d)}$

We can always find $\Delta^*, $B^* if these equations are linearly independent.

In [9]:
delta_star = (Cu - Cd)/(S0*(u-d))
B_star = (u*Cd - d*Cu)/r*(u-d)

It must follow, through the no-arbitrage condition, that

$$
C_0 = \Delta^* S_0 + B^* =
\frac{1}{r}
\left[
\frac{r-d}{u-d}C_u
+
\frac{u-r}{u-d}C_d
\right]
$$

In [10]:
C0 = delta_star * S0 + B_star

A multi-period generalisation exists:

$$
C_0 =
\frac{1}{r^N}
\sum_{k=0}^{N}
\binom{N}{k}
q^k(1-q)^{N-k}
\max\left(S_0u^kd^{N-k}-K,\,0\right)
$$

Where:
$$
q = \frac{r-d}{u-d}
$$

Function below uses backwards induction, working through the tree in reverse until the time-0 option value is obtained. 

In [ ]:
def binomial_price(S0, K, u, d, r, n):

    q = (r - d)/(u - d)

    Sn_list = []

    for k in range(n+1):
        Sn = u**k * d**(n-k) * S0
        Sn_list.append(Sn)

    Cn_list = []

    for i in range(len(Sn_list)):
        Cn = max(Sn_list[i] - K, 0)
        Cn_list.append(Cn)

    current_values = Cn_list

    for k in range(1,n + 1):

        previous_values = []
        
        for i in range(len(Cn_list) - k):
            x = 1/r * (q * (current_values[i+1]) + (1-q) * (current_values[i]))
            previous_values.append(x)

        current_values = previous_values

    return current_values

Binomial pricing gives the discrete-time foundation; Brownian motion and GBM extend this to continuous time, leading to Black–Scholes.